##HOW TO USE:::


WARNING! THIS IS AN OLD VERSION, AND IS NOT AS GOOD AS THE NEW VERSION!. 
This markdown document is NOT intended to be run top-to-bottom. Instead, you must run (in order):

1. The import and install statements
2. The global variable declaration
3. The class structure definition
4. The CSV import cell -- used to inject Google Trends data into our model
5. The cell with the dataset you intend to use (regional OR national--whichever was run most recently is the one used in the model)
6. The model declaration cell

Once you have run these, you can tune the parameters of cell 2 (global variable delcaration) and rerun it as many times as you want to see different model results.

Please enjoy.

In [ ]:
%%capture
!pip install -q pyomo
!pip install cartopy
import cartopy
import pyomo.environ as pyo
import itertools
import sys
import os
import random
from datetime import date
import matplotlib.pyplot as plt
import networkx as nx
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
import pandas as pd
from google.colab import files


if 'google.colab' in sys.modules:
    !pip install idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

In [ ]:
# Global parameters
BAND_NAME = "Thee Optimizers"
MAX_TOUR_DAYS = 60
HOME_BASE = "Nashville"
TOUR_BUDGET = 5000           # Starting budget in dollars
MAX_DISTANCE = 1000          # Maximum distance per leg if no fuel-tank limiter
FUEL_TANK_CAPACITY = 20      # fuel tank capacity (gallons)
FUEL_EFFICIENCY = 25         # miles per gallon
BAND_POPULARITY = 0.3       # Used as a threshold for venue gatekeeping. IRL it could be some calculated metric (eg. Probability a random person knows the band
NUM_BAND_MEMBERS = 4         # Total members in the band

#When true, some cities are technically unvisitable within a single day. This is a problem with data quantity.
GAS_DIST_LIMITER = False     # True: original fuel-tank model; False: distance-only limiter.

NUM_VEHICLES = np.ceil(NUM_BAND_MEMBERS/5)  # Vehicles required (5 people per vehicle)

if GAS_DIST_LIMITER:
    MAX_DISTANCE = FUEL_TANK_CAPACITY * FUEL_EFFICIENCY

try:
    main()
except:
    pass


In [ ]:
# ============================================================================
# CLASSES
# ============================================================================
class Venue:
    """
    Represents a music venue with associated revenue information.
    """
    def __init__(self, name, ticket_price, capacity,
                 flat_payment=0, ticket_proportion=0.0, popularity_threshold = 0.0):
        """
        Initialize a venue.

        Parameters:
            name (str): Venue name
            ticket_price (float): Price per ticket in dollars
            capacity (int): Maximum number of tickets available
            flat_payment (float): Flat fee paid to the band
            ticket_proportion (float): Fraction of ticket revenue the band retains (0.0-1.0)
            popularity_threshold (float): Gatekeeps whether or not a band can play at a venue. Assume it is the probability that a random person will have heard of the band
        """
        self.name = name
        self.ticket_price = ticket_price
        self.capacity = capacity
        self.flat_payment = flat_payment
        self.ticket_proportion = ticket_proportion
        self.popularity_threshold = popularity_threshold

    def expected_revenue(self, regional_fanbase_strength):
        """
        Calculate expected revenue from performing at this venue.

        Revenue consists of:
          - Flat payment
          - Ticket revenue (ticket_proportion * tickets_sold * ticket_price)
          - Merchandise sales (20% of crowd buys merch at $35)

        Parameters:
            regional_fanbase_strength (float): Band popularity in the city (0.0-1.0)

        Returns:
            float: Total expected revenue in dollars
        """
        tickets_sold = self.capacity * regional_fanbase_strength * BAND_POPULARITY
        ticket_rev = self.ticket_proportion * tickets_sold * self.ticket_price
        merch_rev = tickets_sold * 0.20 * 35  # 20% of crowd buys $35 merch
        return self.flat_payment + ticket_rev + merch_rev

    def __repr__(self):
        return (f"Venue({self.name}, Ticket: ${self.ticket_price}, Cap: {self.capacity}, "
                f"Flat: ${self.flat_payment}, Share: {self.ticket_proportion:.2f})")


class City:
    """
    Represents a city with associated cost information and venues.
    """
    def __init__(self, name, avg_gas_price, toll_costs, parking_costs,
                 avg_meal_cost, avg_hotel_cost, regional_fanbase_strength):
        """
        Initialize a city.

        Parameters:
            avg_gas_price (float): Average gas price in the city.
            toll_costs (float): Toll costs within or to reach the city.
            parking_costs (float): Parking costs in the city.
            avg_meal_cost (float): Average meal cost per band member.
            avg_hotel_cost (float): Average hotel cost per band member.
            regional_fanbase_strength (float): Band popularity in this city (0.0-1.0), used to scale expected pull in venues
        """
        self.name = name
        self.avg_gas_price = avg_gas_price * NUM_VEHICLES #We pay for gas for every vehicle we travel
        self.toll_costs = toll_costs * NUM_VEHICLES
        self.parking_costs = parking_costs * NUM_VEHICLES
        self.avg_meal_cost = avg_meal_cost * NUM_BAND_MEMBERS
        self.avg_hotel_cost = avg_hotel_cost * NUM_BAND_MEMBERS
        self.regional_fanbase_strength = regional_fanbase_strength  # Band popularity in this city
        self.venues = []  # List to hold Venue objects

    def add_venue(self, venue):
        """Add a venue to this city."""
        self.venues.append(venue)

    def total_expected_revenue(self):
        """Sum expected revenue from all venues in this city."""
        return sum(venue.expected_revenue(self.regional_fanbase_strength) for venue in self.venues)

    def __repr__(self):
        return f"City({self.name})"



In [ ]:
# Load the file and skip the first two lines
files.upload()
df = pd.read_csv("geoMap.csv", skiprows=3, header=None)
#Cleaning
# Use regex to extract city name and score
df = df.rename(columns={0: 'Metro'})
df = df.rename(columns={1: 'TrendScore'})
# Convert TrendScore to numeric
df['TrendScore'] = pd.to_numeric(df['TrendScore'])
# Optional: clean city names
df['Metro'] = df['Metro'].str.strip()


#Select Cities out of entire google trends data for our 21 national cities
#Oakland and San Fran are in one metro so use the same score
national_selected_cities = [
    'New York NY', 'Los Angeles CA', 'Nashville TN', 'Austin TX',
    'Chicago IL', 'New Orleans LA', 'Seattle-Tacoma WA', 'Portland OR',
    'Atlanta GA', 'Miami-Ft. Lauderdale FL', 'Boston MA-Manchester NH',
    'Philadelphia PA', 'Detroit MI', 'Minneapolis-St. Paul MN',
    'San Francisco-Oakland-San Jose CA', 'Denver CO', 'Memphis TN',
    'San Francisco-Oakland-San Jose CA',  # Oakland folded into SF metro
    'Cleveland-Akron (Canton) OH', 'Washington DC (Hagerstown MD)',
    'St. Louis MO'
]
#Select cities for the regional tour starting in Fayetteville
regional_cities = [
    'Nashville TN',
    'Memphis TN',
    'Birmingham AL',
    'Atlanta GA',
    'Knoxville TN',
    'Chattanooga TN',
    'Louisville KY',
    'Huntsville-Decatur (Florence) AL',
    'Jackson MS',
    'Ft. Smith-Fayetteville-Springdale-Rogers AR'
]
#Filter DataFrame for both national cities and regional cities
regionaly_filtered_df = df[df['Metro'].isin(regional_cities)]
nationally_filtered_df = df[df['Metro'].isin(national_selected_cities)]
#normalize for selected subset
nationally_filtered_df['Normalized TrendScore'] = (nationally_filtered_df['TrendScore'] - nationally_filtered_df['TrendScore'].min()) / (nationally_filtered_df['TrendScore'].max() - nationally_filtered_df['TrendScore'].min())
regionaly_filtered_df['Normalized TrendScore'] = (regionaly_filtered_df['TrendScore'] - regionaly_filtered_df['TrendScore'].min()) / (regionaly_filtered_df['TrendScore'].max() - regionaly_filtered_df['TrendScore'].min())
print(regionaly_filtered_df.head())
print(nationally_filtered_df.head())



In [ ]:
# ============================================================================
# NATIONAL DATA INITIALIZATION
# ============================================================================
def create_cities_and_venues():
    """
    Create and initialize city and venue data.

    Returns:
        tuple: (cities list, city_data dictionary, city_names list)
    """
    # --- Define cities ---
    new_york      = City("New York City",    avg_gas_price=4.20, toll_costs=15, parking_costs=40,
                         avg_meal_cost=30,   avg_hotel_cost=250, regional_fanbase_strength=0.48)
    los_angeles   = City("Los Angeles",      avg_gas_price=4.80, toll_costs=10, parking_costs=25,
                         avg_meal_cost=28,   avg_hotel_cost=220, regional_fanbase_strength=0.78)
    nashville     = City("Nashville",        avg_gas_price=3.50, toll_costs=5,  parking_costs=15,
                         avg_meal_cost=20,   avg_hotel_cost=150, regional_fanbase_strength=0.28)
    austin        = City("Austin",           avg_gas_price=3.35, toll_costs=2,  parking_costs=15,
                         avg_meal_cost=22,   avg_hotel_cost=160, regional_fanbase_strength=0.90)
    chicago       = City("Chicago",          avg_gas_price=4.10, toll_costs=12, parking_costs=28,
                         avg_meal_cost=25,   avg_hotel_cost=180, regional_fanbase_strength=0.34)
    new_orleans   = City("New Orleans",      avg_gas_price=3.45, toll_costs=3,  parking_costs=15,
                         avg_meal_cost=22,   avg_hotel_cost=155, regional_fanbase_strength=0.26)
    seattle       = City("Seattle",          avg_gas_price=4.30, toll_costs=8,  parking_costs=20,
                         avg_meal_cost=26,   avg_hotel_cost=190, regional_fanbase_strength=0.62)
    portland      = City("Portland",         avg_gas_price=4.15, toll_costs=5,  parking_costs=18,
                         avg_meal_cost=24,   avg_hotel_cost=165, regional_fanbase_strength=1)
    atlanta       = City("Atlanta",          avg_gas_price=3.40, toll_costs=4,  parking_costs=12,
                         avg_meal_cost=18,   avg_hotel_cost=130, regional_fanbase_strength=0.16)
    miami         = City("Miami",            avg_gas_price=3.90, toll_costs=7,  parking_costs=18,
                         avg_meal_cost=28,   avg_hotel_cost=200, regional_fanbase_strength=0.09)
    boston        = City("Boston",           avg_gas_price=4.00, toll_costs=10, parking_costs=25,
                         avg_meal_cost=28,   avg_hotel_cost=210, regional_fanbase_strength=0.38)
    philadelphia  = City("Philadelphia",     avg_gas_price=3.90, toll_costs=8,  parking_costs=20,
                         avg_meal_cost=25,   avg_hotel_cost=170, regional_fanbase_strength=0.36)
    detroit       = City("Detroit",          avg_gas_price=3.80, toll_costs=6,  parking_costs=15,
                         avg_meal_cost=20,   avg_hotel_cost=140, regional_fanbase_strength=0.26)
    minneapolis   = City("Minneapolis",      avg_gas_price=3.65, toll_costs=5,  parking_costs=18,
                         avg_meal_cost=23,   avg_hotel_cost=160, regional_fanbase_strength=0.29)
    san_francisco = City("San Francisco",    avg_gas_price=4.78, toll_costs=8, parking_costs=20,
                         avg_meal_cost=30,   avg_hotel_cost=200, regional_fanbase_strength=0.5)
    denver        = City("Denver",           avg_gas_price=3.16, toll_costs=4, parking_costs=15,
                         avg_meal_cost=25,   avg_hotel_cost=178, regional_fanbase_strength=0.41)
    memphis       = City("Memphis",          avg_gas_price=2.79, toll_costs=0,  parking_costs=10,
                     avg_meal_cost=25,       avg_hotel_cost=210, regional_fanbase_strength=0.0)
    oakland       = City("Oakland",          avg_gas_price=5.04, toll_costs=7,  parking_costs=20,
                     avg_meal_cost=20,       avg_hotel_cost=164, regional_fanbase_strength=0.5)
    cleveland     = City("Cleveland",        avg_gas_price=2.90, toll_costs=12, parking_costs=15,
                     avg_meal_cost=30,       avg_hotel_cost=189, regional_fanbase_strength=0.28)
    washington    = City("Washington",    avg_gas_price=3.27, toll_costs=3,  parking_costs=20,
                     avg_meal_cost=25,       avg_hotel_cost=219, regional_fanbase_strength=0.22)
    st_louis      = City("St. Louis",        avg_gas_price=2.96, toll_costs=3,  parking_costs=20,
                     avg_meal_cost=35,       avg_hotel_cost=179, regional_fanbase_strength=0.28)


    # Collect for return
    cities     = [new_york, los_angeles, nashville, austin, chicago, new_orleans,
                  seattle, portland, atlanta, miami, boston, philadelphia,
                  detroit, minneapolis, san_francisco, denver, memphis,
                  oakland, cleveland, washington, st_louis]
    city_data  = {city.name: city for city in cities}
    city_names = [city.name for city in cities]

    # --- Add venues for each city ---

    # New York City
    new_york.add_venue(Venue("Madison Square Garden",       ticket_price=150, capacity=20000,
                              flat_payment=0, ticket_proportion=0.30, popularity_threshold=0.85))
    new_york.add_venue(Venue("Mercury Lounge",              ticket_price=25,  capacity=250,
                              flat_payment=200, ticket_proportion=0.50, popularity_threshold=0.05))
    new_york.add_venue(Venue("Bowery Ballroom",             ticket_price=45,  capacity=575,
                              flat_payment=500, ticket_proportion=0.35, popularity_threshold=0.15))
    new_york.add_venue(Venue("Radio City Music Hall",       ticket_price=95,  capacity=6000,
                              flat_payment=1000, ticket_proportion=0.28, popularity_threshold=0.45))

    # Los Angeles
    los_angeles.add_venue(Venue("Hollywood Bowl",            ticket_price=120, capacity=17500,
                                 flat_payment=10000, ticket_proportion=0.25, popularity_threshold=0.80))
    los_angeles.add_venue(Venue("The Smell",                  ticket_price=20,  capacity=150,
                                 flat_payment=100, ticket_proportion=0.50, popularity_threshold=0.05))
    los_angeles.add_venue(Venue("Teragram Ballroom",          ticket_price=35,  capacity=600,
                                 flat_payment=300, ticket_proportion=0.30, popularity_threshold=0.12))
    los_angeles.add_venue(Venue("Greek Theatre",             ticket_price=75,  capacity=5900,
                                 flat_payment=1500, ticket_proportion=0.25, popularity_threshold=0.40))

    # Nashville
    nashville.add_venue(Venue("Ryman Auditorium",           ticket_price=95,  capacity=5000,
                               flat_payment=0, ticket_proportion=0.30, popularity_threshold=0.70))
    nashville.add_venue(Venue("The End",                     ticket_price=20,  capacity=150,
                               flat_payment=100, ticket_proportion=0.50, popularity_threshold=0.05))
    nashville.add_venue(Venue("Exit/In",                     ticket_price=30,  capacity=500,
                               flat_payment=200, ticket_proportion=0.35, popularity_threshold=0.10))
    nashville.add_venue(Venue("Ascend Amphitheater",         ticket_price=80,  capacity=6800,
                               flat_payment=2500, ticket_proportion=0.20, popularity_threshold=0.45))

    # Austin
    austin.add_venue(Venue("Moody Center",                ticket_price=85,  capacity=15000,
                            flat_payment=5000, ticket_proportion=0.20, popularity_threshold=0.75))
    austin.add_venue(Venue("Hole in the Wall",            ticket_price=15,  capacity=120,
                            flat_payment=75, ticket_proportion=0.60, popularity_threshold=0.04))
    austin.add_venue(Venue("Mohawk Austin",               ticket_price=40,  capacity=700,
                            flat_payment=250, ticket_proportion=0.30, popularity_threshold=0.10))
    austin.add_venue(Venue("ACL Live at Moody Theater",   ticket_price=70,  capacity=2750,
                            flat_payment=1000, ticket_proportion=0.25, popularity_threshold=0.38))

    # Chicago
    chicago.add_venue(Venue("United Center",               ticket_price=110, capacity=23000,
                            flat_payment=0, ticket_proportion=0.25, popularity_threshold=0.85))
    chicago.add_venue(Venue("The Hideout",                 ticket_price=20,  capacity=150,
                            flat_payment=100, ticket_proportion=0.50, popularity_threshold=0.06))
    chicago.add_venue(Venue("Lincoln Hall",                ticket_price=35,  capacity=500,
                            flat_payment=300, ticket_proportion=0.40, popularity_threshold=0.13))
    chicago.add_venue(Venue("Aragon Ballroom",             ticket_price=70,  capacity=5000,
                            flat_payment=2000, ticket_proportion=0.25, popularity_threshold=0.35))

    # New Orleans
    new_orleans.add_venue(Venue("Smoothie King Center",      ticket_price=80,  capacity=16000,
                                 flat_payment=2000, ticket_proportion=0.22, popularity_threshold=0.75))
    new_orleans.add_venue(Venue("Siberia",                    ticket_price=15,  capacity=120,
                                 flat_payment=100, ticket_proportion=0.50, popularity_threshold=0.05))
    new_orleans.add_venue(Venue("Tipitina’s",                 ticket_price=35,  capacity=600,
                                 flat_payment=250, ticket_proportion=0.35, popularity_threshold=0.12))
    new_orleans.add_venue(Venue("The Joy Theater",            ticket_price=60,  capacity=1200,
                                 flat_payment=800, ticket_proportion=0.28, popularity_threshold=0.30))

    # Seattle
    seattle.add_venue(Venue("Climate Pledge Arena",       ticket_price=105, capacity=18000,
                              flat_payment=0, ticket_proportion=0.28, popularity_threshold=0.80))
    seattle.add_venue(Venue("The Sunset",                  ticket_price=18,  capacity=200,
                              flat_payment=100, ticket_proportion=0.50, popularity_threshold=0.06))
    seattle.add_venue(Venue("Neumos",                      ticket_price=40,  capacity=750,
                              flat_payment=300, ticket_proportion=0.30, popularity_threshold=0.14))
    seattle.add_venue(Venue("The Moore Theatre",           ticket_price=65,  capacity=1400,
                              flat_payment=900, ticket_proportion=0.25, popularity_threshold=0.32))

    # Portland
    portland.add_venue(Venue("Moda Center",                ticket_price=100, capacity=19500,
                              flat_payment=5000, ticket_proportion=0.24, popularity_threshold=0.78))
    portland.add_venue(Venue("The Know",                    ticket_price=15,  capacity=100,
                              flat_payment=80, ticket_proportion=0.50, popularity_threshold=0.05))
    portland.add_venue(Venue("Doug Fir Lounge",            ticket_price=35,  capacity=500,
                              flat_payment=250, ticket_proportion=0.35, popularity_threshold=0.11))
    portland.add_venue(Venue("Crystal Ballroom",           ticket_price=60,  capacity=1500,
                              flat_payment=800, ticket_proportion=0.25, popularity_threshold=0.28))

    # Atlanta
    atlanta.add_venue(Venue("State Farm Arena",           ticket_price=100, capacity=21000,
                              flat_payment=0, ticket_proportion=0.25, popularity_threshold=0.80))
    atlanta.add_venue(Venue("529 Bar",                     ticket_price=20,  capacity=100,
                              flat_payment=100, ticket_proportion=0.60, popularity_threshold=0.04))
    atlanta.add_venue(Venue("The Earl",                    ticket_price=35,  capacity=300,
                              flat_payment=200, ticket_proportion=0.35, popularity_threshold=0.10))
    atlanta.add_venue(Venue("Variety Playhouse",           ticket_price=60,  capacity=1100,
                              flat_payment=600, ticket_proportion=0.28, popularity_threshold=0.30))

    # Miami
    miami.add_venue(Venue("Kaseya Center",               ticket_price=95,  capacity=19600,
                            flat_payment=3000, ticket_proportion=0.20, popularity_threshold=0.78))
    miami.add_venue(Venue("Gramps",                       ticket_price=20,  capacity=200,
                            flat_payment=150, ticket_proportion=0.45, popularity_threshold=0.06))
    miami.add_venue(Venue("The Ground",                   ticket_price=40,  capacity=600,
                            flat_payment=250, ticket_proportion=0.32, popularity_threshold=0.14))
    miami.add_venue(Venue("The Fillmore Miami",           ticket_price=70,  capacity=2700,
                            flat_payment=1200, ticket_proportion=0.26, popularity_threshold=0.35))

    # Boston
    boston.add_venue(Venue("TD Garden",                   ticket_price=115, capacity=19000,
                             flat_payment=0, ticket_proportion=0.27, popularity_threshold=0.82))
    boston.add_venue(Venue("Great Scott",                  ticket_price=20,  capacity=150,
                             flat_payment=100, ticket_proportion=0.50, popularity_threshold=0.05))
    boston.add_venue(Venue("Paradise Rock Club",          ticket_price=45,  capacity=725,
                             flat_payment=300, ticket_proportion=0.35, popularity_threshold=0.13))
    boston.add_venue(Venue("House of Blues",              ticket_price=75,  capacity=2400,
                             flat_payment=1500, ticket_proportion=0.25, popularity_threshold=0.40))

    # Philadelphia
    philadelphia.add_venue(Venue("Wells Fargo Center",       ticket_price=90,  capacity=21000,
                                  flat_payment=4000, ticket_proportion=0.23, popularity_threshold=0.78))
    philadelphia.add_venue(Venue("Johnny Brenda’s",           ticket_price=25,  capacity=200,
                                  flat_payment=150, ticket_proportion=0.45, popularity_threshold=0.06))
    philadelphia.add_venue(Venue("Union Transfer",            ticket_price=50,  capacity=1200,
                                  flat_payment=400, ticket_proportion=0.30, popularity_threshold=0.16))
    philadelphia.add_venue(Venue("The Fillmore Philly",       ticket_price=85,  capacity=2600,
                                  flat_payment=1200, ticket_proportion=0.22, popularity_threshold=0.38))

    # Detroit
    detroit.add_venue(Venue("Little Caesars Arena",       ticket_price=85,  capacity=20000,
                               flat_payment=0, ticket_proportion=0.22, popularity_threshold=0.75))
    detroit.add_venue(Venue("Psycho Suzi’s Motor Lounge", ticket_price=15,  capacity=150,
                               flat_payment=100, ticket_proportion=0.50, popularity_threshold=0.05))
    detroit.add_venue(Venue("El Club",                    ticket_price=25,  capacity=300,
                               flat_payment=200, ticket_proportion=0.35, popularity_threshold=0.10))
    detroit.add_venue(Venue("St. Andrew’s Hall",          ticket_price=60,  capacity=1200,
                               flat_payment=800, ticket_proportion=0.28, popularity_threshold=0.32))

    # Minneapolis
    minneapolis.add_venue(Venue("Target Center",            ticket_price=95,  capacity=20000,
                                  flat_payment=2000, ticket_proportion=0.24, popularity_threshold=0.80))
    minneapolis.add_venue(Venue("7th St Entry",              ticket_price=20,  capacity=250,
                                  flat_payment=150, ticket_proportion=0.50, popularity_threshold=0.05))
    minneapolis.add_venue(Venue("First Avenue",             ticket_price=45,  capacity=1500,
                                  flat_payment=500, ticket_proportion=0.35, popularity_threshold=0.14))
    minneapolis.add_venue(Venue("Fillmore Minneapolis",      ticket_price=75,  capacity=2600,
                                  flat_payment=1500, ticket_proportion=0.22, popularity_threshold=0.38))

    # San Francisco
    san_francisco.add_venue(Venue("Chase Center",          ticket_price=125, capacity=18000,
                                 flat_payment=0, ticket_proportion=0.30, popularity_threshold=0.85))
    san_francisco.add_venue(Venue("Slim’s",                ticket_price=35, capacity=500,
                                 flat_payment=300, ticket_proportion=0.35, popularity_threshold=0.12))
    san_francisco.add_venue(Venue("Bill Graham Civic Auditorium", ticket_price=230, capacity=8500,
                                 flat_payment=1000, ticket_proportion=0.30, popularity_threshold=0.30))
    san_francisco.add_venue(Venue("The Warfield",          ticket_price=116, capacity=2300,
                                 flat_payment=800, ticket_proportion=0.28, popularity_threshold=0.35))

    # Denver
    denver.add_venue(Venue("Ball Arena",                   ticket_price=90, capacity=20000,
                                 flat_payment=0, ticket_proportion=0.26, popularity_threshold=0.85))
    denver.add_venue(Venue("Bluebird Theater",             ticket_price=35, capacity=550,
                                 flat_payment=300, ticket_proportion=0.35, popularity_threshold=0.12))
    denver.add_venue(Venue("Mission Ballroom",             ticket_price=65, capacity=3000,
                                 flat_payment=1000, ticket_proportion=0.28, popularity_threshold=0.36))
    denver.add_venue(Venue("Red Rocks",                    ticket_price=139, capacity=9500,
                                 flat_payment=1000, ticket_proportion=0.30, popularity_threshold=0.84))

    # Memphis
    memphis.add_venue(Venue("Orpheum Theatre",             ticket_price=70, capacity=2400,
                                 flat_payment=1000, ticket_proportion=0.35, popularity_threshold=0.70))
    memphis.add_venue(Venue("Hi-Tone Cafe",                ticket_price=25, capacity=300,
                                 flat_payment=200, ticket_proportion=0.35, popularity_threshold=0.10))
    memphis.add_venue(Venue("FedExForum",                  ticket_price=135, capacity=19000,
                                 flat_payment=1200, ticket_proportion=0.30, popularity_threshold=0.80))
    memphis.add_venue(Venue("Levitt Shell",                ticket_price=100, capacity=2500,
                                 flat_payment=700, ticket_proportion=0.28, popularity_threshold=0.62))

    # Oakland
    oakland.add_venue(Venue("Oracle Arena",                ticket_price=80, capacity=19500,
                                 flat_payment=0, ticket_proportion=0.22, popularity_threshold=0.75))
    oakland.add_venue(Venue("Starline Social Club",        ticket_price=30, capacity=400,
                                 flat_payment=200, ticket_proportion=0.35, popularity_threshold=0.10))
    oakland.add_venue(Venue("Fox Theater Oakland",         ticket_price=60, capacity=2800,
                                 flat_payment=1200, ticket_proportion=0.25, popularity_threshold=0.38))
    oakland.add_venue(Venue("Oakland Arena",               ticket_price=200, capacity=19000,
                                 flat_payment=1000, ticket_proportion=0.30, popularity_threshold=0.60))

    # Cleveland
    cleveland.add_venue(Venue("Rocket Mortgage FieldHouse", ticket_price=95, capacity=19700,
                                 flat_payment=0, ticket_proportion=0.25, popularity_threshold=0.80))
    cleveland.add_venue(Venue("Mahall’s",                  ticket_price=25, capacity=350,
                                 flat_payment=200, ticket_proportion=0.35, popularity_threshold=0.12))
    cleveland.add_venue(Venue("Beachland Ballroom",        ticket_price=55, capacity=1200,
                                 flat_payment=800, ticket_proportion=0.28, popularity_threshold=0.30))
    cleveland.add_venue(Venue("Jacobs Pavilion at Nautica", ticket_price=190, capacity=5000,
                                 flat_payment=1200, ticket_proportion=0.30, popularity_threshold=0.48))

    # Washington, D.C.
    washington.add_venue(Venue("Capital One Arena",        ticket_price=100, capacity=20000,
                                 flat_payment=0, ticket_proportion=0.27, popularity_threshold=0.82))
    washington.add_venue(Venue("Black Cat",                ticket_price=20, capacity=200,
                                 flat_payment=150, ticket_proportion=0.50, popularity_threshold=0.05))
    washington.add_venue(Venue("9:30 Club",                ticket_price=45, capacity=1200,
                                 flat_payment=500, ticket_proportion=0.35, popularity_threshold=0.20))
    washington.add_venue(Venue("The Anthem",               ticket_price=75, capacity=6500,
                                 flat_payment=1500, ticket_proportion=0.25, popularity_threshold=0.40))

    # St. Louis
    st_louis.add_venue(Venue("Enterprise Center",          ticket_price=85, capacity=19000,
                                 flat_payment=0, ticket_proportion=0.24, popularity_threshold=0.78))
    st_louis.add_venue(Venue("Delmar Hall",                ticket_price=30, capacity=800,
                                 flat_payment=300, ticket_proportion=0.35, popularity_threshold=0.12))
    st_louis.add_venue(Venue("Hollywood Casino Ampitheatre", ticket_price=100, capacity=20000,
                                 flat_payment=1000, ticket_proportion=0.26, popularity_threshold=0.75))
    st_louis.add_venue(Venue("Stifel Theatre",             ticket_price=122, capacity=3100,
                                 flat_payment=900, ticket_proportion=0.25, popularity_threshold=0.30))


    return cities, city_data, city_names



def create_distance_matrix(city_names):
    """
    Create distance matrix between cities.

    Parameters:
        city_names (list): List of city names

    Returns:
        dict: Distance matrix with (city1, city2) tuples as keys and distances as values
    """
    # Raw nested distances data
    distances = {
        "New York City": {
            "Boston": 215, "Philadelphia": 95,  "Washington": 225,  "Cleveland": 462,
            "Detroit": 614,      "Chicago": 790,     "Nashville": 880,    "Atlanta": 866,
            "Miami": 1280,       "New Orleans": 1300,"Memphis": 1096,     "St. Louis": 954,
            "Minneapolis": 1197, "Denver": 1780,     "Austin": 1742,      "Los Angeles": 2790,
            "San Francisco": 2900,"Oakland": 2900,   "Portland": 2897,    "Seattle": 2852,
        },
        "Boston": {
            "New York City": 215, "Philadelphia": 306, "Washington": 440,  "Cleveland": 640,
            "Detroit": 715,       "Chicago": 983,      "Nashville": 1091,  "Atlanta": 1080,
            "Miami": 1490,        "New Orleans": 1507, "Memphis": 1307,     "St. Louis": 1147,
            "Minneapolis": 1390,  "Denver": 1971,      "Austin": 1950,      "Los Angeles": 3008,
            "San Francisco": 3095,"Oakland": 3095,    "Portland": 3096,    "Seattle": 3051,
        },
        "Philadelphia": {
            "New York City": 95,  "Boston": 306,      "Washington": 140,  "Cleveland": 415,
            "Detroit": 570,       "Chicago": 738,     "Nashville": 828,    "Atlanta": 778,
            "Miami": 1192,        "New Orleans": 1221,"Memphis": 1015,     "St. Louis": 873,
            "Minneapolis": 1115,  "Denver": 1702,     "Austin": 1667,      "Los Angeles": 2710,
            "San Francisco": 2811,"Oakland": 2811,    "Portland": 2815,    "Seattle": 2783,
        },
        "Washington": {
            "New York City": 225, "Boston": 440,      "Philadelphia": 140,"Cleveland": 370,
            "Detroit": 525,       "Chicago": 700,     "Nashville": 670,    "Atlanta": 638,
            "Miami": 1052,        "New Orleans": 1085,"Memphis": 879,      "St. Louis": 837,
            "Minneapolis": 1080,  "Denver": 1651,     "Austin": 1530,      "Los Angeles": 2650,
            "San Francisco": 2802,"Oakland": 2802,    "Portland": 2826,    "Seattle": 2756,
        },
        "Cleveland": {
            "New York City": 462, "Boston": 640,      "Philadelphia": 415,"Washington": 370,
            "Detroit": 170,       "Chicago": 345,     "Nashville": 550,    "Atlanta": 720,
            "Miami": 1235,        "New Orleans": 985, "Memphis": 774,      "St. Louis": 554,
            "Minneapolis": 744,   "Denver": 1364,     "Austin": 1359,      "Los Angeles": 2342,
            "San Francisco": 2443,"Oakland": 2443,    "Portland": 2433,    "Seattle": 2409,
        },
        "Detroit": {
            "New York City": 614, "Boston": 715,      "Philadelphia": 570,"Washington": 525,
            "Cleveland": 170,     "Chicago": 283,     "Nashville": 534,    "Atlanta": 748,
            "Miami": 1380,        "New Orleans": 1033,"Memphis": 790,      "St. Louis": 513,
            "Minneapolis": 694,   "Denver": 1274,     "Austin": 1340,      "Los Angeles": 2281,
            "San Francisco": 2383,"Oakland": 2383,    "Portland": 2373,    "Seattle": 2349,
        },
        "Chicago": {
            "New York City": 790, "Boston": 983,      "Philadelphia": 738,"Washington": 700,
            "Cleveland": 345,     "Detroit": 283,     "Nashville": 472,    "Atlanta": 717,
            "Miami": 1380,        "New Orleans": 925, "Memphis": 530,      "St. Louis": 297,
            "Minneapolis": 409,   "Denver": 1003,     "Austin": 1123,      "Los Angeles": 2017,
            "San Francisco": 2127,"Oakland": 2127,    "Portland": 2119,    "Seattle": 2052,
        },
        "Nashville": {
            "New York City": 880, "Boston": 1091,     "Philadelphia": 828,"Washington": 670,
            "Cleveland": 550,     "Detroit": 534,     "Chicago": 472,      "Atlanta": 250,
            "Miami": 878,         "New Orleans": 532, "Memphis": 210,      "St. Louis": 309,
            "Minneapolis": 809,   "Denver": 1099,     "Austin": 845,       "Los Angeles": 1988,
            "San Francisco": 2289,"Oakland": 2289,    "Portland": 2349,    "Seattle": 2370,
        },
        "Atlanta": {
            "New York City": 866, "Boston": 1080,     "Philadelphia": 778,"Washington": 638,
            "Cleveland": 720,     "Detroit": 748,     "Chicago": 717,      "Nashville": 250,
            "Miami": 661,         "New Orleans": 469, "Memphis": 380,      "St. Louis": 555,
            "Minneapolis": 1074,  "Denver": 1403,     "Austin": 930,       "Los Angeles": 2174,
            "San Francisco": 2471,"Oakland": 2471,    "Portland": 2636,    "Seattle": 2636,
        },
        "Miami": {
            "New York City": 1280,"Boston": 1490,     "Philadelphia": 1192,"Washington": 1052,
            "Cleveland": 1235,    "Detroit": 1380,    "Chicago": 1380,     "Nashville": 878,
            "Atlanta": 661,       "New Orleans": 865, "Memphis": 980,      "St. Louis": 1158,
            "Minneapolis": 1850,  "Denver": 2065,     "Austin": 1311,      "Los Angeles": 2733,
            "San Francisco": 3114,"Oakland": 3114,    "Portland": 3322,    "Seattle": 3295,
        },
        "New Orleans": {
            "New York City": 1300,"Boston": 1507,     "Philadelphia": 1221,"Washington": 1085,
            "Cleveland": 985,     "Detroit": 1033,    "Chicago": 925,      "Nashville": 532,
            "Atlanta": 469,       "Miami": 865,       "Memphis": 395,      "St. Louis": 673,
            "Minneapolis": 1200,  "Denver": 1272,     "Austin": 510,       "Los Angeles": 1889,
            "San Francisco": 2239,"Oakland": 2239,    "Portland": 2485,    "Seattle": 2561,
        },
        "Memphis": {
            "New York City": 1096,"Boston": 1307,     "Philadelphia": 1015,"Washington": 879,
            "Cleveland": 774,     "Detroit": 790,     "Chicago": 530,      "Nashville": 210,
            "Atlanta": 380,       "Miami": 980,       "New Orleans": 395,  "St. Louis": 283,
            "Minneapolis": 790,   "Denver": 1020,     "Austin": 630,       "Los Angeles": 1792,
            "San Francisco": 2088,"Oakland": 2088,    "Portland": 2232,    "Seattle": 2253,
        },
        "St. Louis": {
            "New York City": 954, "Boston": 1147,     "Philadelphia": 873, "Washington": 837,
            "Cleveland": 554,     "Detroit": 513,     "Chicago": 297,      "Nashville": 309,
            "Atlanta": 555,       "Miami": 1158,      "New Orleans": 673,  "Memphis": 283,
            "Minneapolis": 564,   "Denver": 850,      "Austin": 835,       "Los Angeles": 1835,
            "San Francisco": 2080,"Oakland": 2080,    "Portland": 2146,    "Seattle": 2166,
        },
        "Minneapolis": {
            "New York City": 1197,"Boston": 1390,     "Philadelphia": 1115,"Washington": 1080,
            "Cleveland": 744,     "Detroit": 694,     "Chicago": 409,      "Nashville": 809,
            "Atlanta": 1074,      "Miami": 1850,      "New Orleans": 1200,"Memphis": 790,
            "St. Louis": 564,     "Denver": 915,      "Austin": 1175,      "Los Angeles": 1940,
            "San Francisco": 1960,"Oakland": 1960,    "Portland": 1726,    "Seattle": 1656,
        },
        "Denver": {
            "New York City": 1780,"Boston": 1971,     "Philadelphia": 1702,"Washington": 1651,
            "Cleveland": 1364,    "Detroit": 1274,    "Chicago": 1003,     "Nashville": 1099,
            "Atlanta": 1403,      "Miami": 2065,      "New Orleans": 1272,"Memphis": 1020,
            "St. Louis": 850,     "Minneapolis": 915, "Austin": 925,       "Los Angeles": 1015,
            "San Francisco": 1250,"Oakland": 1250,    "Portland": 1248,    "Seattle": 1331,
        },
        "Austin": {
            "New York City": 1742,"Boston": 1950,     "Philadelphia": 1667,"Washington": 1530,
            "Cleveland": 1359,    "Detroit": 1340,    "Chicago": 1123,     "Nashville": 845,
            "Atlanta": 930,       "Miami": 1311,      "New Orleans": 510,  "Memphis": 630,
            "St. Louis": 835,     "Minneapolis": 1175,"Denver": 925,       "Los Angeles": 1377,
            "San Francisco": 1750,"Oakland": 1750,    "Portland": 2064,    "Seattle": 2127,
        },
        "Los Angeles": {
            "New York City": 2790,"Boston": 3008,     "Philadelphia": 2710,"Washington": 2650,
            "Cleveland": 2342,    "Detroit": 2281,    "Chicago": 2017,     "Nashville": 1988,
            "Atlanta": 2174,      "Miami": 2733,      "New Orleans": 1889,"Memphis": 1792,
            "St. Louis": 1835,    "Minneapolis": 1940,"Denver": 1015,      "Austin": 1377,
            "San Francisco": 380, "Oakland": 380,     "Portland": 963,     "Seattle": 1136,
        },
        "San Francisco": {
            "New York City": 2900,"Boston": 3095,     "Philadelphia": 2811,"Washington": 2802,
            "Cleveland": 2443,    "Detroit": 2383,    "Chicago": 2127,     "Nashville": 2289,
            "Atlanta": 2471,      "Miami": 3114,      "New Orleans": 2239,"Memphis": 2088,
            "St. Louis": 2080,    "Minneapolis": 1960,"Denver": 1250,      "Austin": 1750,
            "Los Angeles": 380,   "Oakland": 13,      "Portland": 635,     "Seattle": 808,
        },
        "Oakland": {
            "New York City": 2900,"Boston": 3095,     "Philadelphia": 2811,"Washington": 2802,
            "Cleveland": 2443,    "Detroit": 2383,    "Chicago": 2127,     "Nashville": 2289,
            "Atlanta": 2471,      "Miami": 3114,      "New Orleans": 2239,"Memphis": 2088,
            "St. Louis": 2080,    "Minneapolis": 1960,"Denver": 1250,      "Austin": 1750,
            "Los Angeles": 380,   "San Francisco": 13, "Portland": 635,     "Seattle": 808,
        },
        "Portland": {
            "New York City": 2897,"Boston": 3096,     "Philadelphia": 2815,"Washington": 2826,
            "Cleveland": 2433,    "Detroit": 2373,    "Chicago": 2119,     "Nashville": 2349,
            "Atlanta": 2636,      "Miami": 3322,      "New Orleans": 2485,"Memphis": 2232,
            "St. Louis": 2146,    "Minneapolis": 1726,"Denver": 1248,      "Austin": 2064,
            "Los Angeles": 963,   "San Francisco": 635, "Oakland": 635,     "Seattle": 174,
        },
        "Seattle": {
            "New York City": 2852,"Boston": 3051,     "Philadelphia": 2783,"Washington": 2756,
            "Cleveland": 2409,    "Detroit": 2349,    "Chicago": 2052,     "Nashville": 2370,
            "Atlanta": 2636,      "Miami": 3295,      "New Orleans": 2561,"Memphis": 2253,
            "St. Louis": 2166,    "Minneapolis": 1656,"Denver": 1331,      "Austin": 2127,
            "Los Angeles": 1136,  "San Francisco": 808, "Oakland": 808,     "Portland": 174,
        },
    }

    # Flatten into a (city1, city2) → distance mapping
    distance_matrix = {}
    for city1, targets in distances.items():
        for city2, d in targets.items():
            distance_matrix[(city1, city2)] = d

    # Make the matrix symmetric and add diagonal zeros
    for (i, j), d in list(distance_matrix.items()):
        distance_matrix[(j, i)] = d
    for city in city_names:
        distance_matrix[(city, city)] = 0

    return distance_matrix

def create_city_coords():
    # City coordinates (longitude, latitude)
    city_coords = {
        "New York City": (-74.0060, 40.7128),
        "Los Angeles": (-118.2437, 34.0522),
        "Nashville": (-86.7816, 36.1627),
        "Austin": (-97.7431, 30.2672),
        "Chicago": (-87.6298, 41.8781),
        "New Orleans": (-90.0715, 29.9511),
        "Seattle": (-122.3321, 47.6062),
        "Portland": (-122.6765, 45.5231),
        "Atlanta": (-84.3880, 33.7490),
        "Miami": (-80.1918, 25.7617),
        "Boston": (-71.0589, 42.3601),
        "Philadelphia": (-75.1652, 39.9526),
        "Detroit": (-83.0458, 42.3314),
        "Minneapolis": (-93.2650, 44.9778),
        "San Francisco": (-122.4194, 37.7749),
        "Denver": (-104.9903, 39.7392),
        "Memphis": (-90.0490, 35.1495),
        "Oakland": (-122.2712, 37.8044),
        "Cleveland": (-81.6944, 41.4993),
        "Washington": (-77.0369, 38.9072),
        "St. Louis": (-90.1994, 38.6270)
    }
    return city_coords


In [ ]:
# ============================================================================
# REGIONAL DATA INITIALIZATION
# ============================================================================


#This is smaller-scale data for the American South used to test the model. Feel free to use this instead of the national data.



def create_cities_and_venues():
    """
    Create and initialize city and venue data with popularity thresholds.

    Returns:
        tuple: (cities list, city_data dictionary, city_names list)
    """
    # Create cities with their properties
    nashville   = City("Nashville",   avg_gas_price=3.5, toll_costs=0,  parking_costs=8,
                       avg_meal_cost=35, avg_hotel_cost=150, regional_fanbase_strength=.76)
    memphis     = City("Memphis",     avg_gas_price=3.4, toll_costs=2,  parking_costs=6,
                       avg_meal_cost=30, avg_hotel_cost=120, regional_fanbase_strength=0.12)
    birmingham  = City("Birmingham",  avg_gas_price=2.8, toll_costs=2,  parking_costs=7,
                       avg_meal_cost=32, avg_hotel_cost=130, regional_fanbase_strength=0.32)
    atlanta     = City("Atlanta",     avg_gas_price=2.8, toll_costs=3,  parking_costs=9,
                       avg_meal_cost=38, avg_hotel_cost=160, regional_fanbase_strength=0.48)
    knoxville   = City("Knoxville",   avg_gas_price=3.1, toll_costs=1,  parking_costs=5,
                       avg_meal_cost=28, avg_hotel_cost=110, regional_fanbase_strength=0.44)
    chattanooga = City("Chattanooga", avg_gas_price=3.5, toll_costs=1,  parking_costs=5,
                       avg_meal_cost=30, avg_hotel_cost=115, regional_fanbase_strength=0.56)
    louisville  = City("Louisville",  avg_gas_price=3.6, toll_costs=2,  parking_costs=6,
                       avg_meal_cost=29, avg_hotel_cost=125, regional_fanbase_strength=0.76)
    huntsville  = City("Huntsville",  avg_gas_price=3.0, toll_costs=1,  parking_costs=4,
                       avg_meal_cost=27, avg_hotel_cost=105, regional_fanbase_strength=0.28)
    jackson     = City("Jackson",     avg_gas_price=3.5, toll_costs=1,  parking_costs=5,
                       avg_meal_cost=26, avg_hotel_cost=100, regional_fanbase_strength=0.0)
    fayetteville= City("Fayetteville",avg_gas_price=2.9, toll_costs=0,  parking_costs=8,
                       avg_meal_cost=18, avg_hotel_cost=125, regional_fanbase_strength=1)

    cities = [nashville, memphis, birmingham, atlanta, knoxville,
              chattanooga, louisville, huntsville, jackson, fayetteville]

    city_data = {city.name: city for city in cities}
    city_names = [city.name for city in cities]

    # Add venues (real-world, 3 per city)
    nashville.add_venue(Venue("Ryman Auditorium", 95, 5000, 0, 0.30, 0.60))
    nashville.add_venue(Venue("Exit/In", 40, 500, 250, 0.20, 0.15))
    nashville.add_venue(Venue("Brooklyn Bowl Nashville", 65, 1200, 500, 0.25, 0.35))

    memphis.add_venue(Venue("FedExForum", 90, 6000, 5000, 0.10, 0.70))
    memphis.add_venue(Venue("Hi-Tone Cafe", 30, 350, 150, 0.25, 0.10))
    memphis.add_venue(Venue("Minglewood Hall", 55, 1200, 1000, 0.20, 0.30))

    birmingham.add_venue(Venue("Legacy Arena", 85, 4000, 2000, 0.20, 0.65))
    birmingham.add_venue(Venue("Saturn Birmingham", 40, 500, 300, 0.25, 0.18))
    birmingham.add_venue(Venue("Iron City Bham", 65, 1300, 700, 0.20, 0.40))

    atlanta.add_venue(Venue("State Farm Arena", 100, 8000, 0, 0.25, 0.75))
    atlanta.add_venue(Venue("The Masquerade", 45, 1000, 400, 0.25, 0.25))
    atlanta.add_venue(Venue("Variety Playhouse", 60, 1100, 750, 0.20, 0.35))

    knoxville.add_venue(Venue("Knox Coliseum", 80, 3500, 1000, 0.15, 0.50))
    knoxville.add_venue(Venue("The Mill & Mine", 45, 1200, 400, 0.25, 0.28))
    knoxville.add_venue(Venue("Open Chord Stage", 35, 300, 200, 0.20, 0.12))

    chattanooga.add_venue(Venue("MacArthur Park Amphitheater", 75, 3000, 0, 0.30, 0.55))
    chattanooga.add_venue(Venue("Songbirds", 40, 400, 150, 0.25, 0.15))
    chattanooga.add_venue(Venue("Walker Theatre", 60, 900, 500, 0.20, 0.32))

    louisville.add_venue(Venue("KFC Yum! Center", 95, 5000, 3000, 0.10, 0.70))
    louisville.add_venue(Venue("Headliners Music Hall", 50, 600, 300, 0.20, 0.20))
    louisville.add_venue(Venue("Zanzabar", 30, 300, 150, 0.25, 0.10))

    huntsville.add_venue(Venue("Von Braun Center", 85, 4500, 500, 0.20, 0.65))
    huntsville.add_venue(Venue("Mars Music Hall", 55, 1500, 700, 0.20, 0.30))
    huntsville.add_venue(Venue("The Orion Amphitheater", 80, 8000, 2000, 0.15, 0.50))

    jackson.add_venue(Venue("Jackson Convention Center", 80, 4000, 0, 0.30, 0.55))
    jackson.add_venue(Venue("Duling Hall", 40, 350, 150, 0.25, 0.12))
    jackson.add_venue(Venue("Hal & Mal's", 35, 300, 100, 0.20, 0.08))

    fayetteville.add_venue(Venue("George's Majestic Lounge", 35, 700, 300, 0.25, 0.10))
    fayetteville.add_venue(Venue("JJ's Live", 35, 1000, 1000, 0.20, 0.30))
    fayetteville.add_venue(Venue("Nomad's Trailside", 5, 200, 0, 1, 0))


    return cities, city_data, city_names



def create_distance_matrix(city_names):
    """
    Create distance matrix between cities.

    Parameters:
        city_names (list): List of city names

    Returns:
        dict: Distance matrix with (city1, city2) tuples as keys and distances as values
    """
    # Initialize the distance matrix with key city pairs
    distance_matrix = {
    ("Nashville", "Memphis"): 210,
    ("Nashville", "Birmingham"): 190,
    ("Nashville", "Atlanta"): 250,
    ("Nashville", "Knoxville"): 180,
    ("Nashville", "Chattanooga"): 130,
    ("Nashville", "Louisville"): 175,
    ("Nashville", "Huntsville"): 110,
    ("Nashville", "Jackson"): 220,
    ("Nashville", "Fayetteville"): 385,

    ("Memphis", "Birmingham"): 220,
    ("Memphis", "Atlanta"): 300,
    ("Memphis", "Knoxville"): 250,
    ("Memphis", "Chattanooga"): 150,
    ("Memphis", "Louisville"): 220,
    ("Memphis", "Huntsville"): 130,
    ("Memphis", "Jackson"): 80,
    ("Memphis", "Fayetteville"): 310,

    ("Birmingham", "Atlanta"): 150,
    ("Birmingham", "Knoxville"): 220,
    ("Birmingham", "Chattanooga"): 170,
    ("Birmingham", "Louisville"): 205,
    ("Birmingham", "Huntsville"): 60,
    ("Birmingham", "Jackson"): 150,
    ("Birmingham", "Fayetteville"): 445,

    ("Atlanta", "Knoxville"): 180,
    ("Atlanta", "Chattanooga"): 110,
    ("Atlanta", "Louisville"): 275,
    ("Atlanta", "Huntsville"): 100,
    ("Atlanta", "Jackson"): 250,
    ("Atlanta", "Fayetteville"): 580,

    ("Knoxville", "Chattanooga"): 80,
    ("Knoxville", "Louisville"): 180,
    ("Knoxville", "Huntsville"): 150,
    ("Knoxville", "Jackson"): 240,
    ("Knoxville", "Fayetteville"): 530,

    ("Chattanooga", "Louisville"): 190,
    ("Chattanooga", "Huntsville"): 100,
    ("Chattanooga", "Jackson"): 160,
    ("Chattanooga", "Fayetteville"): 460,

    ("Louisville", "Huntsville"): 210,
    ("Louisville", "Jackson"): 230,
    ("Louisville", "Fayetteville"): 510,

    ("Huntsville", "Jackson"): 140,
    ("Huntsville", "Fayetteville"): 390,

    ("Jackson", "Fayetteville"): 300
}


    # Make the matrix symmetric and add diagonal zeros
    for (i, j), d in list(distance_matrix.items()):
        distance_matrix[(j, i)] = d
    for city in city_names:
        distance_matrix[(city, city)] = 0

    return distance_matrix

def create_city_coords():
    """
    Returns a dict mapping each city name to its (longitude, latitude).
    Used by visualize_tour_route to plot points on the map.
    """
    return {
        "Nashville":    (-86.7816, 36.1627),
        "Memphis":      (-90.04898, 35.14953),
        "Birmingham":   (-86.80249, 33.52066),
        "Atlanta":      (-84.38800, 33.74900),
        "Knoxville":    (-83.92074, 35.96064),
        "Chattanooga":  (-85.30968, 35.04563),
        "Louisville":   (-85.75846, 38.25267),
        "Huntsville":   (-86.58610, 34.73037),
        "Jackson":      (-90.18481, 32.29876),
        "Fayetteville": (-94.15743, 36.06258),  # Fayetteville, Arkansas
    }





In [ ]:
# =============================================================================
# MODEL DEFINITION
# =============================================================================
def create_optimization_model(cities, city_data, distance_matrix):
    model = pyo.ConcreteModel()

    # Sets
    model.CITIES = pyo.Set(initialize=[c.name for c in cities])
    model.EDGES = pyo.Set(
        initialize=[(i, j) for i in model.CITIES for j in model.CITIES if i != j],
        dimen=2
    )

    # Parameters
    model.distance = pyo.Param(model.EDGES,
        initialize=lambda m, i, j: distance_matrix[(i, j)]
    )

    # Decision vars
    model.x = pyo.Var(model.EDGES, domain=pyo.Binary)
    model.y = pyo.Var(model.CITIES, domain=pyo.Binary)
    model.days = pyo.Var(model.CITIES, domain=pyo.NonNegativeIntegers)
    model.u = pyo.Var(model.CITIES, domain=pyo.NonNegativeIntegers, bounds=(0, len(model.CITIES)))
    # fuel consumed arriving at each city
    model.f = pyo.Var(model.CITIES, domain=pyo.NonNegativeReals)

    # Venue selection
    venue_list = [(city.name, v.name) for city in cities for v in city.venues]
    model.VENUES = pyo.Set(initialize=venue_list, dimen=2)
    model.z = pyo.Var(model.VENUES, domain=pyo.Binary)

    # Constraints
    # Range constraint
    def max_range_rule(m, i, j):
        if m.distance[i, j] > MAX_DISTANCE:
            return m.x[i, j] == 0
        return pyo.Constraint.Skip
    model.max_range = pyo.Constraint(model.EDGES, rule=max_range_rule)

    # Fuel usage definition
    def fuel_use_rule(m, j):
        return m.f[j] == sum(
            m.distance[i, j] / FUEL_EFFICIENCY * m.x[i, j]
            for i in m.CITIES if i != j
        )
    model.fuel_use = pyo.Constraint(model.CITIES, rule=fuel_use_rule)
    # Tank capacity limiter
    if GAS_DIST_LIMITER:
        model.fuel_limit = pyo.Constraint(
            model.CITIES,
            rule=lambda m, j: m.f[j] <= FUEL_TANK_CAPACITY
        )

    # Routing
    def outflow_rule(m, i):
        if i == HOME_BASE:
            return sum(m.x[i, j] for j in m.CITIES if j != i) == 1
        return sum(m.x[i, j] for j in m.CITIES if j != i) == m.y[i]
    model.outflow = pyo.Constraint(model.CITIES, rule=outflow_rule)

    def inflow_rule(m, j):
        if j == HOME_BASE:
            return sum(m.x[i, j] for i in m.CITIES if i != j) == 1
        return sum(m.x[i, j] for i in m.CITIES if i != j) == m.y[j]
    model.inflow = pyo.Constraint(model.CITIES, rule=inflow_rule)

    def subtour_rule(m, i, j):
        if i != HOME_BASE and j != HOME_BASE and i != j:
            return m.u[i] - m.u[j] + len(model.CITIES)*m.x[i, j] <= len(model.CITIES)-1
        return pyo.Constraint.Skip
    model.subtour = pyo.Constraint(model.EDGES, rule=subtour_rule)

    # City-day linkage
    model.day_visit = pyo.Constraint(model.CITIES, rule=lambda m,c: m.days[c] >= m.y[c])

    # Venue-city constraints
    model.one_venue = pyo.Constraint(
        model.CITIES,
        rule=lambda m,c: sum(m.z[c,v] for (cc,v) in m.VENUES if cc==c) <= 1
    )
    model.venue_link = pyo.Constraint(model.VENUES, rule=lambda m,c,v: m.z[c,v] <= m.y[c])
    model.pop_threshold = pyo.Constraint(
        model.VENUES,
        rule=lambda m,c,v: (m.z[c,v] == 0)
            if BAND_POPULARITY < next(vn for vn in city_data[c].venues if vn.name==v).popularity_threshold
            else pyo.Constraint.Skip
    )

    # Tour days
    model.total_days = pyo.Constraint(expr=sum(model.days[c] for c in model.CITIES) <= MAX_TOUR_DAYS)

    # Cost expressions
    model.toll_parking_total = pyo.Expression(
        expr=sum(
            model.x[i,j]*(city_data[j].toll_costs+city_data[j].parking_costs)
            for (i,j) in model.EDGES
        )
    )

    model.refuel_total = pyo.Expression(
        expr=sum(
            model.f[c] * city_data[c].avg_gas_price
            for c in model.CITIES
        )
    )

    model.staying_total = pyo.Expression(
        expr=sum(
            model.days[c]*(city_data[c].avg_hotel_cost+city_data[c].avg_meal_cost)
            for c in model.CITIES if c!=HOME_BASE
        )
    )

    # Budget
    model.budget = pyo.Constraint(
        expr=model.toll_parking_total + model.refuel_total + model.staying_total <= TOUR_BUDGET
    )

    # Objective
    def obj_rule(m):
        rev = sum(
            m.z[c,v] * next(vn for vn in city_data[c].venues if vn.name==v)
                .expected_revenue(city_data[c].regional_fanbase_strength)
            for (c,v) in m.VENUES
        )
        return rev - (m.toll_parking_total + m.refuel_total + m.staying_total)
    model.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)

    return model, model.toll_parking_total, model.refuel_total, model.staying_total





# ============================================================================
# SOLVING AND RESULT ANALYSIS
# ============================================================================
def solve_model(model, solver_path):
    """
    Solve the optimization model.

    Parameters:
        model (pyo.ConcreteModel): Model to solve
        solver_path (str): Path to the solver executable

    Returns:
        pyo.results.SolverResults: Solution results
    """
    solver = pyo.SolverFactory(solver_path)
    results = solver.solve(model, tee=True)
    return results


def display_results(model, city_data, toll_parking_cost, refuel_total, staying_total):
    """
    Display the optimization results.

    Parameters:
        model (pyo.ConcreteModel): Solved model
        city_data (dict): Dictionary mapping city names to City objects
        toll_parking_cost (function): Function to calculate toll and parking costs
        refuel_total (pyo.Expression): Expression for total refueling costs
        staying_total (pyo.Expression): Expression for total accommodation costs
    """
    print("\n" + "="*80)
    print("OPTIMIZATION RESULTS")
    print("="*80)

    # Display selected tour route
    print("\nTour Route (selected edges):")
    for (i, j) in model.EDGES:
        if pyo.value(model.x[i, j]) > 0.5:
            print(f"  {i} -> {j}")


    # Display visited cities
    print("\nCities visited:")
    for c in model.CITIES:
        if pyo.value(model.y[c]) > 0.5:
            print(f"  {c}")

    # Display days spent in each city
    print("\nDays spent in each city:")
    for c in model.CITIES:
        days = pyo.value(model.days[c])
        if days > 0:
            print(f"  {c}: {days} days")

    # Display selected venues and their expected revenue
    print("\nSelected Venues:")
    for (c, v) in model.VENUES:
        if pyo.value(model.z[c, v]) > 0.5:
            venue_obj = next(venue for venue in city_data[c].venues if venue.name == v)
            revenue = venue_obj.expected_revenue(city_data[c].regional_fanbase_strength)
            print(f"  {c}: {v} (Expected Revenue: ${revenue:.2f})")

    # Display fuel levels

    if GAS_DIST_LIMITER:
        print("\nFuel remaining on arrival (gallons):")
        for c in model.CITIES:
            if pyo.value(model.y[c]) > 0.5 or c == HOME_BASE:
                print(f"  {c}: {pyo.value(model.f[c]):.2f}")

    # Display cost summary
    total_tp_cost = pyo.value(toll_parking_cost(model))
    total_rf_cost = pyo.value(refuel_total)
    total_st_cost = pyo.value(staying_total)
    total_expenses = total_tp_cost + total_rf_cost + total_st_cost

    print("\nCost Summary:")
    print(f"  Total Toll+Parking Cost: ${total_tp_cost:.2f}")
    print(f"  Total Refuel Cost: ${total_rf_cost:.2f}")
    print(f"  Total Staying Cost: ${total_st_cost:.2f}")
    print(f"  Total Expenses: ${total_expenses:.2f}")
    print(f"  Net Profit: ${pyo.value(model.obj):.2f}")


def reconstruct_tour_route(model):
    """
    Reconstruct the ordered tour route.

    Parameters:
        model (pyo.ConcreteModel): Solved model

    Returns:
        list: Ordered list of (from_city, to_city) tuples
    """
    # Build dictionary of selected arcs
    arc_dict = {}
    for (i, j) in model.EDGES:
        if pyo.value(model.x[i, j]) > 0.5:
            arc_dict[i] = j

    # Reconstruct ordered route
    ordered_route = []
    current_city = HOME_BASE
    while True:
        next_city = arc_dict.get(current_city)
        if next_city is None:
            break
        ordered_route.append((current_city, next_city))
        if next_city == HOME_BASE:
            break
        current_city = next_city

    # Display the route
    print("\nOrdered Tour:")
    if ordered_route:
      print(" -> ".join([i for (i, j) in ordered_route] + [ordered_route[-1][1]]))
    else:
      print("No feasible tour route found.") # Print a message if no route is found

    return ordered_route


def simulate_tour(model, ordered_route, city_data):
    """
    Simulate the tour to show budget flow and earnings for each leg.

    Parameters:
        model (pyo.ConcreteModel): Solved model
        ordered_route (list): Ordered list of (from_city, to_city) tuples
        city_data (dict): Dictionary mapping city names to City objects
    """

    print("\n" + "="*100)
    print("TOUR SIMULATION (Budget Flow and Earnings)")
    print("="*100)

    initial_budget = TOUR_BUDGET
    cumulative_cost = 0.0
    cumulative_revenue = 0.0
    current_budget = initial_budget

    # Helper functions
    def get_toll_parking(city_name):
        return city_data[city_name].toll_costs + city_data[city_name].parking_costs

    def get_refuel_cost(city_name):
        fuel_consumed = pyo.value(model.f[city_name])
        return fuel_consumed * city_data[city_name].avg_gas_price

    def get_staying_cost(city_name):
        if city_name == HOME_BASE:
            return 0.0
        days = pyo.value(model.days[city_name])
        return days * (city_data[city_name].avg_hotel_cost + city_data[city_name].avg_meal_cost)

    def get_city_revenue(city_name):
        for (c, v) in model.VENUES:
            if c == city_name and pyo.value(model.z[c, v]) > 0.5:
                venue_obj = next(venue for venue in city_data[c].venues if venue.name == v)
                return venue_obj.expected_revenue(city_data[c].regional_fanbase_strength)
        return 0.0

    def get_selected_venue(city_name):
        for (c, v) in model.VENUES:
            if c == city_name and pyo.value(model.z[c, v]) > 0.5:
                return v
        return "-"

    # Display header
    print("\nLeg-by-Leg Financial Breakdown:")
    print("-" * 140)
    print(f"{'From':<15} {'To':<15} {'Venue':<35} {'Toll+Park':<10} {'Refuel':<10} "
          f"{'Stay':<10} {'Leg Cost':<10} {'Budget':<10} {'Revenue':<10} {'Cum. Rev.':<10}")
    print("-" * 140)

    prev_city = HOME_BASE
    for (from_city, to_city) in ordered_route:
        toll_cost = get_toll_parking(to_city)
        refuel_cost = get_refuel_cost(to_city)
        stay_cost = get_staying_cost(to_city)
        leg_cost = toll_cost + refuel_cost + stay_cost
        cumulative_cost += leg_cost
        current_budget = initial_budget - cumulative_cost
        city_rev = get_city_revenue(to_city)
        cumulative_revenue += city_rev
        venue_name = get_selected_venue(to_city)

        print(f"{prev_city:<15} {to_city:<15} {venue_name:<35} "
              f"${toll_cost:<9.2f} ${refuel_cost:<9.2f} ${stay_cost:<9.2f} "
              f"${leg_cost:<9.2f} ${current_budget:<9.2f} ${city_rev:<9.2f} ${cumulative_revenue:<9.2f}")

        prev_city = to_city

    print("-" * 140)
    print(f"\nFinal Remaining Budget: ${current_budget:.2f}")
    print(f"Total Cumulative Revenue: ${cumulative_revenue:.2f}")
    print(f"Net Profit (Revenue - Total Expenses): ${cumulative_revenue - cumulative_cost:.2f}")



def generate_tour_timeline(model, ordered_route):
    """
    Generate a timeline of the tour schedule.

    Parameters:
        model (pyo.ConcreteModel): Solved model
        ordered_route (list): Ordered list of (from_city, to_city) tuples
    """
    print("\n" + "="*80)
    print("TOUR SCHEDULE TIMELINE")
    print("="*80)

    if not ordered_route:
      print("No fesible tour route found. Cannot generate timeline.")
      return

    # For this timeline, we assume the tour starts on day 1 and that in each visited city
    # the concert (if any) is played on the arrival day.
    current_day = 1

    # Construct a list of visited cities in order
    visited_cities = [ordered_route[0][0]] + [to_city for (_, to_city) in ordered_route]

    for city in visited_cities:
        arrival_day = current_day
        days_spent = pyo.value(model.days[city])
        print(f"{city} visited on day {arrival_day} (staying for {days_spent} day(s))")

        # Check if there's a performance at this city
        for (c, v) in model.VENUES:
            if c == city and pyo.value(model.z[c, v]) > 0.5:
                # Assume the concert is played on the arrival day
                print(f"  Performance at '{v}' on day {arrival_day}\n")

        current_day += days_spent


# Function to create and display the tour map
def visualize_tour_route(ordered_route, city_data, city_venues=None, save_path=None):
    """
    Visualize a tour route on a US map.

    Parameters:
        ordered_route (list): List of (from_city, to_city) tuples representing the route
        city_data (dict): Dictionary with city data including coordinates
        city_venues (dict, optional): Dictionary mapping cities to their selected venues
        save_path (str, optional): Path to save the figure
    """

    city_coords = create_city_coords()

    # Create figure with cartopy projection
    fig = plt.figure(figsize=(15, 10))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.LambertConformal(
        central_longitude=-95, central_latitude=37.5))

    # Set map extent to continental US
    ax.set_extent([-125, -66.5, 24, 50], ccrs.PlateCarree())

    # Add map features
    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.STATES, linestyle=':')

    # Create a directed graph for the tour
    G = nx.DiGraph()

    # Extract cities from ordered route
    cities_in_route = []
    for from_city, to_city in ordered_route:
        if from_city not in cities_in_route:
            cities_in_route.append(from_city)
        if to_city not in cities_in_route:
            cities_in_route.append(to_city)
        G.add_edge(from_city, to_city)

    # Plot cities with different sizes based on if they're in the route
    for city, (lon, lat) in city_coords.items():
        if city in cities_in_route:
              # Add city name
              ax.text(lon + 0.5, lat, city, transform=ccrs.PlateCarree(),
                  fontsize=9, ha='left', va='center', zorder=6,
                  bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.2'))


              if city == HOME_BASE:
                  # Home base is big and red
                  ax.plot(lon, lat, '*', transform=ccrs.PlateCarree(),
                  markersize=12, color='black', alpha=1, zorder=3)

              else:
                  # Cities  in route are red
                  ax.plot(lon, lat, 'o', transform=ccrs.PlateCarree(),
                        markersize=8, color='red', alpha=0.5, zorder=3)

        else:
            # Cities not in route are gray
            ax.plot(lon, lat, 'o', transform=ccrs.PlateCarree(),
                   markersize=12, color='gray', alpha=0.5, zorder=3)

    # Draw edges with arrows for direction
    # Use a colormap to show the order of the route
    cmap = plt.cm.viridis
    color = 'red'

    for i, (from_city, to_city) in enumerate(ordered_route):
        from_lon, from_lat = city_coords[from_city]
        to_lon, to_lat = city_coords[to_city]

        # Draw the edge with an appropriate color
        ax.plot([from_lon, to_lon], [from_lat, to_lat], '-',
                transform=ccrs.PlateCarree(), linewidth=2,
                color=color, zorder=4)

        # Add an arrow to show direction
        # Calculate the midpoint of the line
        mid_lon = (from_lon + to_lon) / 2
        mid_lat = (from_lat + to_lat) / 2

        # Calculate the direction vector
        dx = to_lon - from_lon
        dy = to_lat - from_lat

        # Normalize and scale
        length = np.sqrt(dx**2 + dy**2)
        dx = dx / length
        dy = dy / length

        # Draw the arrow at the midpoint
        ax.arrow(mid_lon - dx * 0.2, mid_lat - dy * 0.2,
                dx * 0.4, dy * 0.4,
                transform=ccrs.PlateCarree(),
                head_width=0.3, head_length=0.3,
                fc='black', ec='black', zorder=4)

    # Add legend
    legend_elements = [
        Line2D([0], [0], marker='o', color='red', label='Cities in Tour',
               markerfacecolor='red', markersize=10),
        Line2D([0], [0], marker='o', color='gray', label='Other Cities',
               markerfacecolor='gray', markersize=6, alpha=0.5)
    ]

    # Add venues to the legend if provided
    if city_venues:
        venue_info = []
        for city in cities_in_route:
            if city in city_venues and city_venues[city]:
                venue_info.append(f"{city}: {city_venues[city]}")

        if venue_info:
            venue_text = "\n".join(venue_info)
            props = dict(boxstyle='round', facecolor='wheat', alpha=0.7)
            ax.text(0.05, 0.05, "Venues:\n" + venue_text, transform=ax.transAxes,
                   fontsize=9, verticalalignment='bottom', bbox=props)

    ax.legend(handles=legend_elements, loc='lower right')

    # Add title and grid
    plt.title('Optimized Tour Route', fontsize=16)
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()
    return fig, ax

#  function to extract data from model results
def extract_tour_data(model):

    # Build a dict of all chosen arcs
    selected_edges = [
        (i, j) for (i,j) in model.EDGES
        if pyo.value(model.x[i,j]) > 0.5
    ]
    arc_dict = dict(selected_edges)

    # **Force-start** from the real global HOME_BASE
    current_city = HOME_BASE
    ordered_route = []

    # Walk the route until you get back or run out of arcs
    while True:
        next_city = arc_dict.get(current_city)
        if next_city is None:
            break
        ordered_route.append((current_city, next_city))
        if next_city == HOME_BASE:
            break
        current_city = next_city

    # Grab whichever venues you picked
    city_venues = {
        c: v for (c,v) in model.VENUES
        if pyo.value(model.z[c, v]) > 0.5
    }

    return ordered_route, city_venues



# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    """Main execution function."""
    # Initialize data
    cities, city_data, city_names = create_cities_and_venues()
    distance_matrix = create_distance_matrix(city_names)

    # Create and solve model
    model, toll_parking_cost, refuel_total, staying_total = create_optimization_model(
        cities, city_data, distance_matrix
    )

    solver_path = '/content/bin/cbc'  # Path to solver
    results = solve_model(model, solver_path)

    # Check if solution is feasible
    if results.solver.termination_condition == pyo.TerminationCondition.infeasible:
        print("The model is infeasible.")
        return

    else:
    # Display and analyze results
        display_results(model, city_data, toll_parking_cost, refuel_total, staying_total)
        ordered_route = reconstruct_tour_route(model)
        simulate_tour(model, ordered_route, city_data)
        generate_tour_timeline(model, ordered_route)
        ordered_route, city_venues = extract_tour_data(model)
        visualize_tour_route(ordered_route, city_data, city_venues)



if __name__ == "__main__":
    main()

##EXPERIMENTAL CLASS STRUCTURE

This could be modified to fit into our model. We did not get time to impliment this fully, but here is the basic gist.

In [ ]:

# === City Class ===
class City:
    def __init__(self, name, state, avg_gas_price, toll_costs, parking_costs, avg_meal_cost, avg_hotel_cost, regional_fanbase_strength):
        self.name = name
        self.state = state
        self.avg_gas_price = avg_gas_price
        self.toll_costs = toll_costs
        self.parking_costs = parking_costs
        self.avg_meal_cost = avg_meal_cost
        self.avg_hotel_cost = avg_hotel_cost
        self.regional_fanbase_strength = regional_fanbase_strength  # Default pull for that city
        self.venues = []

    def add_venue(self, name, ticket_price, capacity, available_dates,
                  popularity_threshold, flat_payment=0, ticket_proportion=0.0):
        """
        Creates a Venue, links it with this city, and adds it to the list.
        """
        venue = Venue(
            name=name,
            city=self,
            ticket_price=ticket_price,
            capacity=capacity,
            available_dates=available_dates,
            popularity_threshold=popularity_threshold,
            flat_payment=flat_payment,
            ticket_proportion=ticket_proportion
        )
        self.venues.append(venue)
        return venue

    def __repr__(self):
        return f"City({self.name}, {self.state})"


# === Venue Class ===
class Venue:
    def __init__(self, name, city, ticket_price, capacity, available_dates,
                 popularity_threshold, flat_payment=0, ticket_proportion=0.0):
        self.name = name
        self.city = city  # Link back to the City instance
        self.ticket_price = ticket_price
        self.capacity = capacity
        self.available_dates = available_dates  # List of available datetime.date objects
        self.popularity_threshold = popularity_threshold  # Minimum band popularity required (0 to 1)
        self.flat_payment = flat_payment
        self.ticket_proportion = ticket_proportion  # Proportion of ticket revenue paid to the band

    def is_compatible(self, band_popularity):
        """
        A venue is compatible if the band's popularity is at or above the venue's threshold.
        """
        return band_popularity >= self.popularity_threshold

    def expected_revenue(self, regional_fanbase_strength=None):
        """
        Calculates expected revenue, taking into account:
          - A flat payment.
          - A share of the ticket revenue.
          - Merchandise revenue (assumed at 20% of the sold crowd spending $35).
        If no regional_fanbase_strength is provided, the city's fanbase strength is used.
        """
        if regional_fanbase_strength is None:
            regional_fanbase_strength = self.city.regional_fanbase_strength

        tickets_sold = self.capacity * regional_fanbase_strength
        ticket_rev = self.ticket_proportion * tickets_sold * self.ticket_price
        merch_rev = tickets_sold * 0.20 * 35
        return self.flat_payment + ticket_rev + merch_rev

    def __repr__(self):
        return (f"Venue({self.name}, City: {self.city.name}, Ticket: {self.ticket_price}, "
                f"Cap: {self.capacity}, Flat: {self.flat_payment}, Share: {self.ticket_proportion}, "
                f"MinPop: {self.popularity_threshold})")


# === Day Class ===
class Day:
    def __init__(self, date):
        self.date = date  # The specific date of the day.
        self.start_city = None  # City where the day begins.
        self.end_city = None    # City where the day ends.
        self.travel_legs = []   # List of tuples: (from_city, to_city)
        self.shows_played = []  # List of Venue instances where shows occurred.
        self.expenses = 0.0     # Total expenses for the day.
        self.earnings = 0.0     # Total earnings for the day.

    def set_start_city(self, city):
        self.start_city = city

    def set_end_city(self, city):
        self.end_city = city

    def add_travel(self, from_city, to_city):
        """
        Adds a travel leg from one city to another.
        """
        self.travel_legs.append((from_city, to_city))

    def add_show(self, venue):
        """
        Records a show played at a venue. The venue holds its own reference to its city.
        """
        self.shows_played.append(venue)

    def add_expense(self, amount):
        self.expenses += amount

    def add_earning(self, amount):
        self.earnings += amount

    def travel_count(self):
        return len(self.travel_legs)

    def show_count(self):
        return len(self.shows_played)

    def __repr__(self):
        travel_str = " → ".join([f"{frm.name}->{to.name}" for frm, to in self.travel_legs]) or "No Travel"
        show_str = ", ".join([f"{venue.name} ({venue.city.name})" for venue in self.shows_played]) or "No Shows"
        return (
            f"Day({self.date}):\n"
            f"  Start City: {self.start_city.name if self.start_city else 'N/A'}\n"
            f"  Travel Legs: {travel_str}\n"
            f"  Shows Played: {self.show_count()} [{show_str}]\n"
            f"  Expenses: ${self.expenses:.2f} | Earnings: ${self.earnings:.2f}\n"
            f"  End City: {self.end_city.name if self.end_city else 'N/A'}"
        )


# === Band Class ===
class Band:
    def __init__(self, name, popularity, vehicle_range, initial_budget=0.0):
        self.name = name
        self.popularity = popularity    # Float between 0 and 1 indicating band popularity.
        self.vehicle_range = vehicle_range  # Maximum travel distance the band's vehicle can cover
        self.budget = initial_budget
        self.itinerary = []             # List of Day objects representing the tour itinerary

    def plan_day(self, day_date, start_city, end_city, venues,
                 travel_expense, additional_expenses=0.0):
        """
        Create a Day in the itinerary:
          - Set start and end cities.
          - Log travel if the start and end are different.
          - Add the selected venues (if they are available and compatible).
          - Update earnings and expenses.
          - Automatically update the band's budget.
        """
        day = Day(day_date)
        day.set_start_city(start_city)
        if start_city.name != end_city.name:
            day.add_travel(start_city, end_city)
        day.set_end_city(end_city)

        total_show_earnings = 0.0
        # Process each show, using the venue's expected revenue.
        for venue in venues:
            # Check if venue is available on this date
            if day_date in venue.available_dates and venue.is_compatible(self.popularity):
                day.add_show(venue)
                # Calculate revenue based on venue expectations and city's fanbase strength.
                revenue = venue.expected_revenue(venue.city.regional_fanbase_strength)
                total_show_earnings += revenue

        # Add the expenses, which may include travel plus any other incurred costs.
        day.add_expense(travel_expense + additional_expenses)
        day.add_earning(total_show_earnings)
        # Update the band's budget accordingly
        self.budget += total_show_earnings - (travel_expense + additional_expenses)

        # Add the planned day to the itinerary
        self.itinerary.append(day)
        return day

    def show_itinerary(self):
        for day in self.itinerary:
            print(day)
            print("-" * 60)

    def __repr__(self):
        return f"Band({self.name}, Popularity: {self.popularity}, Budget: ${self.budget:.2f})"


# Create cities from the provided list


OG DISTANCE MATRIX